
# Scripted Trade with Heston

1. Heston model recap
2. Multi-Asset Heston calibration
3. Demo 
    - calibration to DAX, STOXX50 and S&P500 options
    - pricing: Vanilla Equity Option, Barrier Option, Rainbow Option with multiple underlyings
    - spot-check: Heston path histogram vs semi-analytic pdf

### Model

Heston model SDE
\begin{align*}
dS &= \mu\,S\,dt + \sqrt{V}\,S\,dW \\
dV &= \kappa\,(\theta - V)\,dt + \sigma\,\sqrt{V}\,dZ \\
dW\,dZ &= \rho\,dt
\end{align*}

Features:
- semi-analytical expressions for European Option prices (available in QuantLib), the bi-variate probability density function $p(s,v)$, and the marginal spot price density $p(s)$ 
- can reproduce volatility smile and skew observed in the market, approximately (see below) 

Five model parameters
- $\theta$, long term variance-level
- $\kappa$, mean reversion speed of the variance process
- $\sigma$, volatility of the variance process
- $\rho$, instantaneous correlation between spot and variance processes
- $v_0 = V(0)$, initial variance

### Feller constraint

Ensure $2\,\theta\,\kappa / \sigma^2 > 1$ to avoid that a material number of variance process paths "get stuck" at zero variance over time and variance probability density piles up near the origin. However, that constraint limits the model's ability to match quoted option prices.

### Calibration 

We currently support the Heston model with constant parameters in ORE, time-dependence to follow.
The model is calibrated to a section of the volatility surface, specified in terms of option expiries and log-moneyness in smile-direction, i.e. strike prices 

$$ K = K_{atm} \times \exp\left(\mbox{moneyness}\times\sigma_{atm}\times\sqrt{\mbox{time-to-expiry}}\right)$$

with at-the-money forward price $K_{atm}$ and volatility $\sigma_{atm}$ for the given option expiry.

We currently support two calibration approaches:

1. Two-Step Approach:
   - Fit $\theta$, $\kappa$ and $v_0$ to the variance curve, currently computed from the volatility surface using functionality in the **replication variance swap engine** in QuantExt (summing option prices across the smile section), and using a simple analytic expression for the Heston-model-implied expected variance.
   - Use the above as initial values for a least-squares fit of all five model parameters, using a gradient search (Levenberg Marquardt)
2. Randomised Initial Values: Here we generate random initial values and perform multiple calibrations using gradient search, keeping the best result. 

### Multi-Asset Heston Model

The ScriptedTrade supports payoffs in multiple assets that all follow individual Heston processes. This requires calibration of the individual Heston models as described above.

Moreover, as in the case of multiple Black processes, we need to specify a correlation matrix that covers all stochastic processes involved. The new challenge with multiple Heston processes is that each Heston model comes with two processes. In addition to the exogeneous spot-spot correlations (e.g. estimated from historical data), we have "internal" Heston model spot-variance correlations (fixed by calibration), and so far unspecified spot-variance and variance-variance correlations across the Heston models for different assets.

We set those assuming **partial independence**, i.e. a dependence structure across models that is induced by the spot-spot correlations $r_{ij}$ across models, i.e. with 2-d block matrices $C_i$ along the diagonal and $C_{ij}$ off-diagonal within the large correlation matrix covering all processes:
$$       
       C_i =\left(\begin{array}{cc} 1 & \rho_i \\ \rho_i & 1 \end{array} \right)
       \qquad
       C_{ij} = r_{ij}\:\left(\begin{array}{cc} 1 & \rho_j \\ \rho_i & \rho_i \,\rho_j \end{array}\right)
$$

With this setup we can generate properly correlated random variates to drive all processes. However, to feed the individual Heston processes, we need to decorrelate the internal spot and variance processes again, since the Heston process in QuantLib expects uncorrelated input variates (say $dW_1$ and $dW_2$), and we want to use the original Heston processes in QuantLib. QuantLib's Heston process constructs correlated variates $d\widetilde W_1=dW_1$ and $d\widetilde W_2=\rho\, dW_1 + \sqrt{1-\rho^2}\,dW_2$, or
$$
d\widetilde W = Q\:dW,\qquad Q = \left(\begin{array}{cc} 1 & 0 \\ \rho & \sqrt{1-\rho^2} \end{array} \right)
$$
which is one way of computing the matrix square root of the correlation matrix $C_i$.

Since we have properly correlated variates $d\widetilde W$ already generated by the "outer" multi-asset process, we need to decorrelate them for consistency with QuantLib using
$$
dW = Q^{-1}\: d\widetilde W, \qquad Q^{-1}\: = \left(\begin{array}{cc} 1 & 0 \\ -\rho/\sqrt{1-\rho^2} & 1/\sqrt{1-\rho^2}\end{array} \right).
$$

### Foreign Equity/Commodity Drift Adjustment

For EQ/COM indices denoted in a currency other than base/domestic, the system of processes will
contain at least one FX index that converts from that foreign into domestic/base currency of the calculation, which is also modelled with a Heston process. The change of measure from the foreign equity's currency into domestic currency then causes drift adjustments
in the foreign equity's Heston process 
\begin{align*}
  \mu(t) &\rightarrow \mu(t) - \rho_{SX}\:\sqrt{V(t)}\:\sqrt{V_X(t)} \\
  \kappa\,\theta &\rightarrow \kappa\,\theta - \rho_{VX}\: \sigma\:\sqrt{V(t)}\:\sqrt{V_X(t)} 
\end{align*}
where 
- $\rho_{SX}$ is the correlation between the EQ/COM spot price process and the FX spot process,
- $\rho_{VX}=\rho_{SX}\,\rho$ is the correlation between the EQ/COM variance process and the FX spot process, and
- $V_X(t)$ is the stochastic variance of the FX process. 


### Modules

In [ ]:
from ORE import *
import sys, time, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.append('..')
import utilities
import cantaro

### Run with Black model

In [ ]:
params = Parameters()
params.fromFile("Input/ore_black.xml")
ore = OREApp(params)
ore.run()
utilities.checkErrorsAndRunTime(ore)

### NPV report

In [ ]:
reportBlack = ore.getReport("npv")
utilities.format_report(reportBlack)

### Run again with Heston calibration

In [ ]:
params = Parameters()
params.fromFile("Input/ore_heston.xml")
ore = OREApp(params)
ore.run()
utilities.checkErrorsAndRunTime(ore)
reportHeston = ore.getReport("npv")
utilities.format_report(reportHeston)

### Calibration reports

In [ ]:
import pandas as pd

trade = "RainbowOption"
#index = "EQ-RIC:.STOXX50E"
#index = "EQ-RIC:.GDAXI"
index = "EQ-RIC:.SPX"
expiry = "3M"

df1 = pd.read_csv('Output/assetmodel_calibration.csv')
df1 = df1[df1["#TradeID"] == trade]
df1 = df1[df1["Index"] == index]
display(df1)

df2 = pd.read_csv('Output/assetmodel_calibration_detail.csv')
df2 = df2[df2["#TradeID"] == trade]
df2 = df2[df2["Index"] == index]
df2 = df2[df2["Expiry"] == expiry]
display(df2)


### Plot smile section

In [ ]:
x = df2["Moneyness"]
y1 = df2["MarketVol"]
y2 = df2["ModelVol"]

plt.plot(x, y1, marker="o", linewidth=2, label='Market Vol')
plt.plot(x, y2, marker="3", linewidth=2, label='Model Vol')

plt.xlabel("Moneyness") 
plt.ylabel("Implied Vol") 
plt.title(index + ' Expiry ' + expiry)
plt.legend()

plt.show()

### Get some additional results

In [ ]:
res = pd.read_csv('Output/additional_results.csv')

# filter the trade
res = res[res["#TradeId"] == trade]
#display(res[res["ResultId"].str.contains("Heston")])
#display(res[res["ResultId"].str.contains("Heston.Index")])
display(res[res["ResultId"].str.contains("Heston.Forward")])


### Create QuantLib Heston Process

... to utilise the Heston PDF 

In [ ]:
time = 0.25

# SPX
spot = 4273.794200
forward = 4320.387965
v0 = 0.017259 
kappa = 8.350790 
theta = 0.034309 
sigma = 1.513896 
rho = -0.703615 

# STOXX
#spot = 4337.500000
#forward = 4371.906927
#v0 = 0.024300
#kappa = 13.977853
#theta = 0.032730
#sigma = 1.912942 
#rho = -0.638495

zeroYield = 0.0
drift = np.log(forward/spot)/time
dividendYield =  drift + zeroYield

Settings.instance().evaluationdate = Date(5, June, 2023)
dayCount = ActualActual(ActualActual.ISDA)
calendar = TARGET()
spotHandle = QuoteHandle(SimpleQuote(spot))
yieldTS = YieldTermStructureHandle(FlatForward(0, calendar, zeroYield, dayCount))
dividendTS = YieldTermStructureHandle(FlatForward(0, calendar, dividendYield, dayCount))
discretization = HestonProcess.QuadraticExponential
process =  HestonProcess(yieldTS, dividendTS, spotHandle, v0, kappa, theta, sigma, rho, discretization)

print("feller:  ", 2*theta*kappa/sigma**2)
print("spot:    ", spot)
print("forward: ", spotHandle.value() * yieldTS.discount(time) / dividendTS.discount(time), forward)
print("ln(spot):", np.log(spot))

# what's wrong with QuantLib's Heston pdf?
# use cantaro86 below instead

### Get path data 

In [ ]:
np.set_printoptions(linewidth=np.inf)

date = "2023-09-05"

# load data frame and filter
res = pd.read_csv('Output/assetmodel_paths.csv')
res = res[res["#TradeId"] == trade]
res = res[res["Date"] == date]

# extract the sample values
data = res[res["Index"] == index]["Value"].to_numpy(dtype="float32")

# convert to log ( s(t) / spot )
data = np.log(data/spot)

print ("data: ", data)
print("length:", len(data))
print("mean:  ", np.mean(data))
print("stdev: ", math.sqrt(np.var(data)))
      

In [ ]:
x = np.linspace(-0.4, 0.2, 200)

plt.hist(data, density=True, bins=100, label="simulation")
plt.plot(x, [cantaro.Heston_pdf(y, t=time, v0=v0, mu=drift, theta=theta, sigma=sigma, kappa=kappa, rho=rho) for y in x], 
         linewidth=3, label="semi-analytic")
plt.xlabel("log(S/S0)") 
plt.ylabel("Density") 
plt.title(index)
plt.xlim(-0.4, 0.2)
plt.legend(loc="upper left")
plt.show()

In [ ]:
trade = "RainbowOption"
date = "2023-09-05"
index = "EQ-RIC:.SPX"
fxIndex = "FX-GENERIC-USD-EUR"
expiry = "3M"
spot = 4273.794200
fwd = 4320.387965
eurDiscount = 0.990574
usdDiscount = 0.986921
fxSpot = 1/1.10025 # 1 USD in EUR
fxFwd = fxSpot * usdDiscount / eurDiscount

eq = res[res["Index"] == index]["Value"].to_numpy(dtype="float32")
fx = res[res["Index"] == fxIndex]["Value"].to_numpy(dtype="float32")

print(len(eq), eq)
print(len(fx), fx)


In [ ]:
assert len(eq) == len(fx)
eqMean = np.mean(eq)
fxMean = np.mean(fx)
price = np.dot(eq, fx) / len(eq)

price0 = fwd * fxFwd
price1 = eqMean * fxMean;
print("E[P(t)] vs fwd:      %10.2f %10.2f %10.2f %+10.1e" % (eqMean, fwd, eqMean - fwd, (eqMean - fwd) / fwd))
print("E[X(t)] vs fxFwd:    %10.6f %10.6f %10.6f %+10.1e" % (fxMean, fxFwd, fxMean - fxFwd, (fxMean - fxFwd) / fxFwd))
print("E[P*X] vs fwd*fxFwd: %10.2f %10.2f %10.2f %+10.1e" % (price, price0, price - price0, (price - price0) / price0))
print("E[P*X] vs E[P]*E[X]: %10.2f %10.2f %10.2f %+10.1e" % (price, price1, price - price1, (price - price1) / price1))
